# BasketData — Limpieza de datos
Fuente: https://www.kaggle.com/datasets/nathanlauga/nba-games

Objetivo: preparar partidos históricos y estadísticas individuales sin inventar datos. Ejecutar desde la raíz del proyecto. Licencia pendiente de comprobar en la ficha original.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
raw = Path("data/raw")
archivos = ["games.csv", "games_details.csv", "teams.csv"]
for nombre in archivos:
    if not (raw / nombre).is_file():
        raise FileNotFoundError(f"Falta {raw / nombre}")


## 1. Inspección
Contar registros originales y valores faltantes antes de transformar. Se exige más de 1.000 registros originales.

In [2]:
g = pd.read_csv(raw / "games.csv", low_memory=False)
t = pd.read_csv(raw / "teams.csv")
cols = ["GAME_ID", "TEAM_ID", "PLAYER_ID", "PLAYER_NAME", "MIN", "PTS", "REB", "AST", "STL", "BLK"]
d = pd.read_csv(raw / "games_details.csv", usecols=cols, low_memory=False)
for nombre, df in [("partidos",g),("equipos",t),("actuaciones",d)]:
    print(nombre, df.shape, "duplicados exactos:", df.duplicated().sum())
    print(df.isna().sum())
assert len(g)>1000

partidos (26651, 21) duplicados exactos: 0
GAME_DATE_EST        0
GAME_ID              0
GAME_STATUS_TEXT     0
HOME_TEAM_ID         0
VISITOR_TEAM_ID      0
SEASON               0
TEAM_ID_home         0
PTS_home            99
FG_PCT_home         99
FT_PCT_home         99
FG3_PCT_home        99
AST_home            99
REB_home            99
TEAM_ID_away         0
PTS_away            99
FG_PCT_away         99
FT_PCT_away         99
FG3_PCT_away        99
AST_away            99
REB_away            99
HOME_TEAM_WINS       0
dtype: int64
equipos (30, 14) duplicados exactos: 0
LEAGUE_ID             0
TEAM_ID               0
MIN_YEAR              0
MAX_YEAR              0
ABBREVIATION          0
NICKNAME              0
YEARFOUNDED           0
CITY                  0
ARENA                 0
ARENACAPACITY         4
OWNER                 0
GENERALMANAGER        0
HEADCOACH             0
DLEAGUEAFFILIATION    0
dtype: int64
actuaciones (668628, 10) duplicados exactos: 285


GAME_ID             0
TEAM_ID             0
PLAYER_ID           0
PLAYER_NAME         0
MIN            109690
REB            109690
AST            109690
STL            109690
BLK            109690
PTS            109690
dtype: int64


## 2. Partidos
Conservar resultados completos y finales. Eliminar duplicados por partido tras ordenar por fecha. No imputar marcadores. Incorporar nombres de equipos mediante identificadores.

In [3]:
g["GAME_DATE_EST"] = pd.to_datetime(g["GAME_DATE_EST"], errors="coerce")
for k in ["PTS_home","PTS_away","AST_home","AST_away","REB_home","REB_away"]:
    g[k] = pd.to_numeric(g[k], errors="coerce")
g = g.loc[g.GAME_STATUS_TEXT.eq("Final")].dropna(subset=["GAME_DATE_EST","GAME_ID","HOME_TEAM_ID","VISITOR_TEAM_ID","SEASON","PTS_home","PTS_away"])
g = g.loc[(g.PTS_home>=0)&(g.PTS_away>=0)&(g.PTS_home!=g.PTS_away)].sort_values("GAME_DATE_EST").drop_duplicates("GAME_ID",keep="last").copy()
mapa=t.drop_duplicates("TEAM_ID").set_index("TEAM_ID")["ABBREVIATION"]
g["local"]=g.HOME_TEAM_ID.map(mapa)
g["visitante"]=g.VISITOR_TEAM_ID.map(mapa)
g=g.dropna(subset=["local","visitante"])
g["ganador"]=np.where(g.PTS_home>g.PTS_away,g.local,g.visitante)
g["puntos_totales"]=g.PTS_home+g.PTS_away
partidos=g[["GAME_ID","GAME_DATE_EST","SEASON","local","visitante","PTS_home","PTS_away","AST_home","AST_away","REB_home","REB_away","ganador","puntos_totales"]].rename(columns={"GAME_ID":"id_partido","GAME_DATE_EST":"fecha","SEASON":"temporada","PTS_home":"puntos_local","PTS_away":"puntos_visitante","AST_home":"asistencias_local","AST_away":"asistencias_visitante","REB_home":"rebotes_local","REB_away":"rebotes_visitante"})

## 3. Juego
Excluir actuaciones sin estadísticas completas o sin minutos jugados; no convertir ausencias en ceros. Conservar las seis pistas y la identidad del jugador. La fecha y el equipo permiten contextualizar las preguntas.

In [4]:
def minutos(x):
    try:
        s=str(x).split(":")
        return float(s[0])+float(s[1])/60 if len(s)==2 else float(s[0])
    except (ValueError, TypeError):
        return np.nan
d["MIN"]=d.MIN.map(minutos)
for k in ["PTS","REB","AST","STL","BLK"]:
    d[k]=pd.to_numeric(d[k], errors="coerce")
d=d.dropna(subset=cols).copy()
d=d.loc[(d.MIN>0)&(d[["PTS","REB","AST","STL","BLK"]]>=0).all(axis=1)].drop_duplicates(["GAME_ID","PLAYER_ID"])
d["equipo"]=d.TEAM_ID.map(mapa)
d=d.merge(partidos[["id_partido","fecha","temporada"]],left_on="GAME_ID",right_on="id_partido",how="inner",validate="many_to_one")
jugadores=d.dropna(subset=["equipo"])[["id_partido","fecha","temporada","PLAYER_ID","PLAYER_NAME","equipo","MIN","PTS","REB","AST","STL","BLK"]].rename(columns={"PLAYER_ID":"id_jugador","PLAYER_NAME":"jugador","MIN":"minutos","PTS":"puntos","REB":"rebotes","AST":"asistencias","STL":"robos","BLK":"bloqueos"})
jugadores["minutos"]=jugadores.minutos.round(2)

## 4. Exportación y verificación
Si el archivo de actuaciones supera 50 MB, conservar un subconjunto reproducible de las temporadas recientes para el juego. Los partidos permanecen independientes.

In [5]:
assert partidos.id_partido.is_unique
assert len(partidos)>1000
assert jugadores.id_jugador.nunique()>=4
partidos.to_csv("data/datos.csv",index=False)
jugadores=jugadores.sort_values("fecha",ascending=False)
jugadores.to_csv("data/jugadores.csv",index=False)
while Path("data/jugadores.csv").stat().st_size>50_000_000:
    jugadores=jugadores.iloc[:int(len(jugadores)*0.8)].copy()
    jugadores.to_csv("data/jugadores.csv",index=False)
for nombre,df in [("datos",partidos),("jugadores",jugadores)]:
    print(nombre,len(df),Path(f"data/{nombre}.csv").stat().st_size/1e6,"MB")
    display(df.head())

datos 26523 1.966031 MB


,id_partido,fecha,temporada,local,visitante,puntos_local,puntos_visitante,asistencias_local,asistencias_visitante,rebotes_local,rebotes_visitante,ganador,puntos_totales
19288,10300001,2003-10-05,2003,UTA,DAL,90.0,85.0,23.0,20.0,41.0,38.0,UTA,175.0
19287,10300002,2003-10-06,2003,MEM,MIL,105.0,94.0,25.0,20.0,48.0,43.0,MEM,199.0
19280,10300010,2003-10-07,2003,WAS,NYK,104.0,86.0,26.0,19.0,45.0,37.0,WAS,190.0
19286,10300009,2003-10-07,2003,SAC,LAC,101.0,82.0,19.0,13.0,39.0,50.0,SAC,183.0
19285,10300005,2003-10-07,2003,POR,HOU,104.0,80.0,22.0,10.0,33.0,37.0,POR,184.0


jugadores 558478 42.095569 MB


,id_partido,fecha,temporada,id_jugador,jugador,equipo,minutos,puntos,rebotes,asistencias,robos,bloqueos
0,22200477,2022-12-22,2022,1629641,Romeo Langford,SAS,18.10,2.0,2.0,0.0,1.0,0.0
38,22200478,2022-12-22,2022,1631117,Walker Kessler,UTA,32.38,12.0,14.0,1.0,0.0,2.0
28,22200478,2022-12-22,2022,1628420,Monte Morris,WAS,30.27,5.0,4.0,4.0,0.0,0.0
29,22200478,2022-12-22,2022,201959,Taj Gibson,WAS,25.03,6.0,4.0,1.0,1.0,0.0
30,22200478,2022-12-22,2022,1629060,Rui Hachimura,WAS,23.58,7.0,7.0,1.0,1.0,0.0
